# SFLLD — Complete Regular and Strict BLS State Fallback (Revised)

This notebook completes the BLS lineage in parallel for **Regular PIT** and **Strict / leakage-safe PIT**.

## Fixed inputs

**Loan-level input**

`data/output/final_modeling_dataset/SFLLD_modeling_dataset_leakage_safe_final.parquet`

**Authoritative monthly State BLS source**

`data/processed/macro_sources/01_primary_modeling/bls/bls_monthly_state_seasonally_adjusted_labor_market_clean.parquet`

The State source was explicitly identified from the project structure and is **not auto-discovered**.

## Core rules

1. Preserve every existing column and raw BLS value.
2. Regular PIT State lookup uses `PIT_BLS_Month_Date`.
3. Strict State lookup reuses the existing 09G `Strict_PIT_BLS_State_Month_Date` rather than substituting a looser timing rule.
4. Reconstruct State 12-month signals by exact calendar-month matching:
   - `Employment_Growth_12M = Employment_t / Employment_(t-12) - 1`
   - `Unemployment_Rate_Change_12M = UR_t - UR_(t-12)`
5. State fallback is used only for geographically comparable signals:
   - unemployment rate,
   - employment growth 12M,
   - unemployment-rate change 12M.
6. State employment and labor-force **totals are never used to fill MSA/MSAD totals**.
7. The newly reconstructed Strict State unemployment rate must reconcile with the existing 09G `_State_BLS_Unemployment_Rate`.
8. Hard QA protects row count, `Loan_ID`, `Target_24M`, `Split`, raw columns, and fallback precedence.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

# ------------------------------------------------------------
# Robust project-root detection
# ------------------------------------------------------------
START = Path.cwd().resolve()

INPUT_REL = (
    Path("data") / "output" / "final_modeling_dataset"
    / "SFLLD_modeling_dataset_leakage_safe_final.parquet"
)

STATE_BLS_REL = (
    Path("data") / "processed" / "macro_sources" / "01_primary_modeling"
    / "bls" / "bls_monthly_state_seasonally_adjusted_labor_market_clean.parquet"
)

ROOT = None
for candidate in [START, *START.parents]:
    if (candidate / INPUT_REL).exists() and (candidate / STATE_BLS_REL).exists():
        ROOT = candidate
        break

if ROOT is None:
    raise FileNotFoundError(
        "Could not locate the SFLLD project root containing BOTH:\n"
        f"  {INPUT_REL}\n"
        f"  {STATE_BLS_REL}\n\n"
        f"Current working directory:\n  {START}"
    )

INPUT = ROOT / INPUT_REL
STATE_BLS_SOURCE = ROOT / STATE_BLS_REL

OUTPUT = (
    ROOT / "data" / "output" / "final_modeling_dataset"
    / "SFLLD_modeling_dataset_complete_regular_and_strict_BLS_state_fallback.parquet"
)

QA_DIR = (
    ROOT / "data" / "output" / "quality_assurance"
    / "10_complete_regular_and_strict_BLS_state_fallback"
)
QA_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook cwd     :", START)
print("Project root     :", ROOT)
print("Loan input       :", INPUT)
print("State BLS source :", STATE_BLS_SOURCE)
print("Output           :", OUTPUT)
print("QA directory     :", QA_DIR)

Notebook cwd     : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\notebooks
Project root     : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6
Loan input       : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_modeling_dataset_leakage_safe_final.parquet
State BLS source : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\processed\macro_sources\01_primary_modeling\bls\bls_monthly_state_seasonally_adjusted_labor_market_clean.parquet
Output           : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_modeling_dataset_complete_regular_and_strict_BLS_state_fallback.parquet
QA directory     : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\quality_assura

In [2]:
# ------------------------------------------------------------
# Load loan-level input and freeze immutable references
# ------------------------------------------------------------
df = pd.read_parquet(INPUT)

print(f"Loaded loan dataset: {df.shape[0]:,} rows × {df.shape[1]:,} columns")

required = [
    "Loan_ID", "Dataset_Vintage", "Target_24M", "Split", "Property_State",
    "PIT_BLS_Month_Date", "Strict_PIT_BLS_Month_Date",
    "Strict_PIT_BLS_State_Month_Date",
    "BLS_Unemployment_Rate", "BLS_Employment", "BLS_Labor_Force",
    "BLS_Employment_Growth_12M", "BLS_Unemployment_Rate_Change_12M",
    "LS_BLS_Unemployment_Rate", "LS_BLS_Employment", "LS_BLS_Labor_Force",
    "LS_BLS_Employment_Growth_12M", "LS_BLS_Unemployment_Rate_Change_12M",
    "_State_FIPS", "_State_BLS_Unemployment_Rate",
    "_State_BLS_Civilian_Labor_Force", "_State_BLS_Employment",
    "LS_BLS_Unemployment_Rate_State_Fallback",
    "LS_BLS_Unemployment_Rate_Final",
    "LS_BLS_Geography_Source_Final",
    "LS_BLS_Fallback_Level",
    "LS_BLS_State_Fallback_Applied",
    "09G_BLS_Remaining_Missing_Reason",
]

missing = [c for c in required if c not in df.columns]
assert not missing, f"Required input columns missing: {missing}"

n0 = len(df)
cols0 = df.columns.tolist()
loan_id0 = df["Loan_ID"].copy()
target0 = df["Target_24M"].copy()
split0 = df["Split"].copy()

# Preserve all original columns for exact post-build checks.
original = df[cols0].copy()

assert df["Loan_ID"].notna().all(), "Loan_ID contains missing values."
assert df["Loan_ID"].is_unique, "Loan_ID is not unique."

for c in ["PIT_BLS_Month_Date", "Strict_PIT_BLS_Month_Date", "Strict_PIT_BLS_State_Month_Date"]:
    df[c] = pd.to_datetime(df[c], errors="coerce")

display(df[required[:15]].head())


Loaded loan dataset: 713,000 rows × 91 columns


,Loan_ID,Dataset_Vintage,Target_24M,Split,Property_State,PIT_BLS_Month_Date,Strict_PIT_BLS_Month_Date,Strict_PIT_BLS_State_Month_Date,BLS_Unemployment_Rate,BLS_Employment,BLS_Labor_Force,BLS_Employment_Growth_12M,BLS_Unemployment_Rate_Change_12M,LS_BLS_Unemployment_Rate,LS_BLS_Employment
0,F10Q10000014,2010.0,0,Other Period,OH,2010-04-01,2010-02-01,2010-02-01,11.5,273584.0,309031.0,0.006267,-0.7,12.0,272332.0
1,F10Q10000069,2010.0,0,Other Period,KS,2010-02-01,2010-02-01,2009-12-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,F10Q10000089,2010.0,0,Other Period,FL,2010-02-01,2009-12-01,2009-12-01,13.9,115822.0,134497.0,-0.048377,2.2,13.7,116629.0
3,F10Q10000098,2010.0,0,Other Period,VA,2010-02-01,2009-12-01,2009-12-01,7.5,777850.0,840511.0,-0.015192,1.1,7.4,775140.0
4,F10Q10000102,2010.0,0,Other Period,KY,2010-03-01,2010-03-01,2010-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. Load and validate the authoritative monthly State BLS source

Expected schema includes:

- `BLS_State_FIPS_Code`
- `BLS_State_Name`
- `BLS_Year`
- `BLS_Month`
- `BLS_Date`
- `BLS_Civilian_Labor_Force`
- `BLS_Employment`
- `BLS_Unemployment`
- `BLS_Unemployment_Rate`
- `BLS_Labor_Force_Participation_Rate`
- `BLS_Employment_Population_Ratio`

The source must be unique by `State FIPS × calendar month`.


In [3]:
state_raw = pd.read_parquet(STATE_BLS_SOURCE)
print(f"Loaded State BLS source: {state_raw.shape[0]:,} rows × {state_raw.shape[1]:,} columns")

state_required = [
    "BLS_State_FIPS_Code", "BLS_State_Name", "BLS_Year", "BLS_Month", "BLS_Date",
    "BLS_Civilian_Labor_Force", "BLS_Employment", "BLS_Unemployment",
    "BLS_Unemployment_Rate", "BLS_Labor_Force_Participation_Rate",
    "BLS_Employment_Population_Ratio", "BLS_Seasonal_Adjustment", "BLS_Geo_Type"
]
state_missing = [c for c in state_required if c not in state_raw.columns]
assert not state_missing, f"Authoritative State BLS source missing columns: {state_missing}"

state = state_raw[state_required].copy()

state["State_FIPS"] = (
    state["BLS_State_FIPS_Code"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.extract(r"(\d+)", expand=False)
    .str.zfill(2)
)

state["Month"] = (
    pd.to_datetime(state["BLS_Date"], errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)

# Cross-check BLS_Date against BLS_Year/BLS_Month.
year_num = pd.to_numeric(state["BLS_Year"], errors="coerce")
month_num = pd.to_numeric(
    state["BLS_Month"].astype("string").str.extract(r"(\d+)", expand=False),
    errors="coerce"
)
ym = pd.to_datetime(
    {"year": year_num, "month": month_num, "day": 1},
    errors="coerce"
)

date_disagree = state["Month"].notna() & ym.notna() & (state["Month"] != ym)
assert not date_disagree.any(), (
    f"BLS_Date disagrees with BLS_Year/BLS_Month on {int(date_disagree.sum()):,} rows."
)

numeric_map = {
    "BLS_Civilian_Labor_Force": "State_BLS_Civilian_Labor_Force",
    "BLS_Employment": "State_BLS_Employment",
    "BLS_Unemployment": "State_BLS_Unemployment",
    "BLS_Unemployment_Rate": "State_BLS_Unemployment_Rate",
    "BLS_Labor_Force_Participation_Rate": "State_BLS_Labor_Force_Participation_Rate",
    "BLS_Employment_Population_Ratio": "State_BLS_Employment_Population_Ratio",
}

for src, dst in numeric_map.items():
    state[dst] = pd.to_numeric(state[src], errors="coerce")

keep = [
    "State_FIPS", "Month", "BLS_State_Name", "BLS_Seasonal_Adjustment", "BLS_Geo_Type",
    *numeric_map.values()
]
state = state[keep].dropna(subset=["State_FIPS", "Month"]).copy()

dup = state.duplicated(["State_FIPS", "Month"], keep=False)
if dup.any():
    display(state.loc[dup].sort_values(["State_FIPS", "Month"]).head(100))
    raise AssertionError(
        f"Authoritative State BLS source has {int(dup.sum()):,} rows participating "
        "in duplicate State_FIPS × Month keys."
    )

state = state.sort_values(["State_FIPS", "Month"]).reset_index(drop=True)

print("Validated State BLS table:", state.shape)
print("Date range:", state["Month"].min(), "to", state["Month"].max())
print("States/FIPS:", state["State_FIPS"].nunique())
display(state.head())


Loaded State BLS source: 31,008 rows × 16 columns
Validated State BLS table: (31008, 11)
Date range: 1976-01-01 00:00:00 to 2026-08-01 00:00:00
States/FIPS: 51


,State_FIPS,Month,BLS_State_Name,BLS_Seasonal_Adjustment,BLS_Geo_Type,State_BLS_Civilian_Labor_Force,State_BLS_Employment,State_BLS_Unemployment,State_BLS_Unemployment_Rate,State_BLS_Labor_Force_Participation_Rate,State_BLS_Employment_Population_Ratio
0,01,1976-01-01,Alabama,SA,STATE,1486509,1387606,98903,6.7,57.1,53.3
1,01,1976-02-01,Alabama,SA,STATE,1485944,1387267,98677,6.6,56.9,53.2
2,01,1976-03-01,Alabama,SA,STATE,1486172,1388303,97869,6.6,56.8,53.1
3,01,1976-04-01,Alabama,SA,STATE,1488773,1392012,96761,6.5,56.8,53.1
4,01,1976-05-01,Alabama,SA,STATE,1492486,1396244,96242,6.4,56.8,53.2


## 2. Build exact calendar-month State 12M signals

A calendar self-join is used rather than `groupby().shift(12)`. This prevents a missing calendar month from accidentally turning the previous available observation into a false 12-month lag.


In [4]:
lag = state[
    ["State_FIPS", "Month", "State_BLS_Employment", "State_BLS_Unemployment_Rate"]
].copy()

# Move t-12 records forward by 12 calendar months so they join to t.
lag["Month"] = lag["Month"] + pd.DateOffset(months=12)
lag = lag.rename(columns={
    "State_BLS_Employment": "State_BLS_Employment_Lag12",
    "State_BLS_Unemployment_Rate": "State_BLS_Unemployment_Rate_Lag12",
})

state = state.merge(
    lag,
    on=["State_FIPS", "Month"],
    how="left",
    validate="one_to_one"
)

state["State_BLS_Employment_Growth_12M"] = np.where(
    state["State_BLS_Employment_Lag12"].notna()
    & (state["State_BLS_Employment_Lag12"] != 0),
    state["State_BLS_Employment"] / state["State_BLS_Employment_Lag12"] - 1.0,
    np.nan
)

state["State_BLS_Unemployment_Rate_Change_12M"] = (
    state["State_BLS_Unemployment_Rate"]
    - state["State_BLS_Unemployment_Rate_Lag12"]
)

display(
    state[
        [
            "State_FIPS", "Month",
            "State_BLS_Employment",
            "State_BLS_Employment_Lag12",
            "State_BLS_Employment_Growth_12M",
            "State_BLS_Unemployment_Rate",
            "State_BLS_Unemployment_Rate_Lag12",
            "State_BLS_Unemployment_Rate_Change_12M",
        ]
    ].head(15)
)


,State_FIPS,Month,State_BLS_Employment,State_BLS_Employment_Lag12,State_BLS_Employment_Growth_12M,State_BLS_Unemployment_Rate,State_BLS_Unemployment_Rate_Lag12,State_BLS_Unemployment_Rate_Change_12M
0,01,1976-01-01,1387606,<NA>,NaN,6.7,<NA>,<NA>
1,01,1976-02-01,1387267,<NA>,NaN,6.6,<NA>,<NA>
2,01,1976-03-01,1388303,<NA>,NaN,6.6,<NA>,<NA>
3,01,1976-04-01,1392012,<NA>,NaN,6.5,<NA>,<NA>
4,01,1976-05-01,1396244,<NA>,NaN,6.4,<NA>,<NA>
5,01,1976-06-01,1400036,<NA>,NaN,6.5,<NA>,<NA>
6,01,1976-07-01,1402922,<NA>,NaN,6.5,<NA>,<NA>
7,01,1976-08-01,1405216,<NA>,NaN,6.7,<NA>,<NA>
8,01,1976-09-01,1407784,<NA>,NaN,6.8,<NA>,<NA>
9,01,1976-10-01,1410640,<NA>,NaN,6.9,<NA>,<NA>


## 3. Canonical loan State key and reusable State lookup

The loan-level `_State_FIPS` generated in the existing pipeline is reused. No new state-name-to-FIPS mapping is invented here.


In [5]:
loan_state_fips = (
    df["_State_FIPS"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.extract(r"(\d+)", expand=False)
    .str.zfill(2)
)

state_value_cols = [
    "State_BLS_Unemployment_Rate",
    "State_BLS_Civilian_Labor_Force",
    "State_BLS_Employment",
    "State_BLS_Unemployment",
    "State_BLS_Labor_Force_Participation_Rate",
    "State_BLS_Employment_Population_Ratio",
    "State_BLS_Employment_Growth_12M",
    "State_BLS_Unemployment_Rate_Change_12M",
]

state_lookup = state[["State_FIPS", "Month", *state_value_cols]].copy()

def lookup_state_bls(base, month_series, prefix):
    keys = pd.DataFrame({
        "_row_id": np.arange(len(base), dtype=np.int64),
        "State_FIPS": loan_state_fips.to_numpy(),
        "Month": pd.to_datetime(month_series, errors="coerce")
                    .dt.to_period("M").dt.to_timestamp().to_numpy(),
    })

    merged = keys.merge(
        state_lookup,
        on=["State_FIPS", "Month"],
        how="left",
        validate="many_to_one"
    ).sort_values("_row_id")

    assert len(merged) == len(base), "State BLS lookup changed row count."
    assert np.array_equal(merged["_row_id"].to_numpy(), np.arange(len(base))), "Lookup changed row order."

    return pd.DataFrame(
        {f"{prefix}{c}": merged[c].to_numpy() for c in state_value_cols},
        index=base.index
    )


## 4. Regular PIT branch

Regular State BLS is looked up at `PIT_BLS_Month_Date`.

The existing Regular MSA/MSAD BLS variables remain untouched. Final variables use:

`existing Regular local value → State value only if local is missing`.


In [6]:
regular_state = lookup_state_bls(df, df["PIT_BLS_Month_Date"], "_REG_")
df = pd.concat([df, regular_state], axis=1)

# Explicit Regular state lineage.
df["BLS_Unemployment_Rate_State_Fallback"] = df["_REG_State_BLS_Unemployment_Rate"]
df["BLS_Employment_Growth_12M_State_Fallback"] = df["_REG_State_BLS_Employment_Growth_12M"]
df["BLS_Unemployment_Rate_Change_12M_State_Fallback"] = df["_REG_State_BLS_Unemployment_Rate_Change_12M"]

# Final geographically comparable signals.
df["BLS_Unemployment_Rate_Final"] = (
    df["BLS_Unemployment_Rate"]
    .fillna(df["BLS_Unemployment_Rate_State_Fallback"])
)
df["BLS_Employment_Growth_12M_Final"] = (
    df["BLS_Employment_Growth_12M"]
    .fillna(df["BLS_Employment_Growth_12M_State_Fallback"])
)
df["BLS_Unemployment_Rate_Change_12M_Final"] = (
    df["BLS_Unemployment_Rate_Change_12M"]
    .fillna(df["BLS_Unemployment_Rate_Change_12M_State_Fallback"])
)

reg_local = df["BLS_Unemployment_Rate"].notna()
reg_state_used = df["BLS_Unemployment_Rate"].isna() & df["BLS_Unemployment_Rate_State_Fallback"].notna()

df["BLS_State_Fallback_Applied"] = reg_state_used.astype("Int8")
df["BLS_Fallback_Level"] = pd.Series(
    np.select([reg_local, reg_state_used], [0, 1], default=2),
    index=df.index,
    dtype="Int8"
)
df["BLS_Geography_Source_Final"] = pd.Series(
    np.select(
        [reg_local, reg_state_used],
        ["MSA/MSAD_REGULAR", "STATE_REGULAR"],
        default="UNRESOLVED"
    ),
    index=df.index,
    dtype="string"
)

df["BLS_Remaining_Missing_Reason"] = pd.Series(
    np.select(
        [
            df["BLS_Unemployment_Rate_Final"].notna(),
            loan_state_fips.isna(),
            df["PIT_BLS_Month_Date"].isna(),
        ],
        [
            "NOT_MISSING",
            "MISSING_STATE_FIPS",
            "MISSING_REGULAR_PIT_MONTH",
        ],
        default="NO_STATE_BLS_VALUE_AT_REGULAR_PIT"
    ),
    index=df.index,
    dtype="string"
)

print("Regular PIT final coverage:")
display(
    pd.DataFrame({
        "raw_missing": [
            df["BLS_Unemployment_Rate"].isna().sum(),
            df["BLS_Employment_Growth_12M"].isna().sum(),
            df["BLS_Unemployment_Rate_Change_12M"].isna().sum(),
        ],
        "final_missing": [
            df["BLS_Unemployment_Rate_Final"].isna().sum(),
            df["BLS_Employment_Growth_12M_Final"].isna().sum(),
            df["BLS_Unemployment_Rate_Change_12M_Final"].isna().sum(),
        ],
    }, index=[
        "Unemployment Rate",
        "Employment Growth 12M",
        "Unemployment Rate Change 12M"
    ])
)


Regular PIT final coverage:


,raw_missing,final_missing
Unemployment Rate,128657,412
Employment Growth 12M,128657,412
Unemployment Rate Change 12M,128657,412


## 5. Strict / leakage-safe branch

**Important:** the Strict State lookup uses the already established 09G field:

`Strict_PIT_BLS_State_Month_Date`

It does **not** replace this with `Strict_PIT_BLS_Month_Date`.

This preserves the existing publication-aware State BLS timing. The reconstructed Strict State unemployment rate is first reconciled against the existing 09G `_State_BLS_Unemployment_Rate`.


In [7]:
strict_state = lookup_state_bls(
    df,
    df["Strict_PIT_BLS_State_Month_Date"],
    "_STRICT_"
)
df = pd.concat([df, strict_state], axis=1)

# ------------------------------------------------------------
# Reconcile with existing 09G strict-state unemployment result
# ------------------------------------------------------------
new_ur = pd.to_numeric(df["_STRICT_State_BLS_Unemployment_Rate"], errors="coerce")
old_ur = pd.to_numeric(df["_State_BLS_Unemployment_Rate"], errors="coerce")

both = new_ur.notna() & old_ur.notna()
value_mismatch = both & ~np.isclose(
    new_ur, old_ur,
    rtol=0.0, atol=1e-10,
    equal_nan=True
)

missingness_mismatch = new_ur.isna() ^ old_ur.isna()

recon = pd.DataFrame({
    "check": [
        "rows",
        "existing_09G_nonmissing",
        "reconstructed_nonmissing",
        "both_nonmissing",
        "value_mismatch_on_both_nonmissing",
        "missingness_mismatch",
    ],
    "value": [
        len(df),
        int(old_ur.notna().sum()),
        int(new_ur.notna().sum()),
        int(both.sum()),
        int(value_mismatch.sum()),
        int(missingness_mismatch.sum()),
    ]
})
display(recon)
recon.to_csv(QA_DIR / "10_strict_state_unemployment_reconstruction.csv", index=False)

if value_mismatch.any() or missingness_mismatch.any():
    bad = value_mismatch | missingness_mismatch
    df.loc[
        bad,
        [
            "Loan_ID", "_State_FIPS",
            "Strict_PIT_BLS_State_Month_Date",
            "_State_BLS_Unemployment_Rate",
            "_STRICT_State_BLS_Unemployment_Rate",
            "BLS_State_Assumed_Available_Date",
        ]
    ].to_csv(
        QA_DIR / "10_strict_state_unemployment_reconstruction_mismatches.csv",
        index=False
    )
    raise AssertionError(
        "Reconstructed Strict State unemployment does not exactly reproduce the existing 09G lineage. "
        "Inspect the mismatch QA file before proceeding."
    )

print("PASS: authoritative source + existing strict State month reproduces 09G State unemployment.")


,check,value
0,rows,713000
1,existing_09G_nonmissing,712639
2,reconstructed_nonmissing,712639
3,both_nonmissing,712639
4,value_mismatch_on_both_nonmissing,0
5,missingness_mismatch,0


PASS: authoritative source + existing strict State month reproduces 09G State unemployment.


In [8]:
# ------------------------------------------------------------
# Complete Strict State fallback for the additional comparable signals
# ------------------------------------------------------------

# Existing 09G unemployment State fallback/final are preserved.
# New state candidates are created for the two additional comparable signals.
df["LS_BLS_Employment_Growth_12M_State_Fallback"] = (
    df["_STRICT_State_BLS_Employment_Growth_12M"]
)
df["LS_BLS_Unemployment_Rate_Change_12M_State_Fallback"] = (
    df["_STRICT_State_BLS_Unemployment_Rate_Change_12M"]
)

df["LS_BLS_Employment_Growth_12M_Final"] = (
    df["LS_BLS_Employment_Growth_12M"]
    .fillna(df["LS_BLS_Employment_Growth_12M_State_Fallback"])
)

df["LS_BLS_Unemployment_Rate_Change_12M_Final"] = (
    df["LS_BLS_Unemployment_Rate_Change_12M"]
    .fillna(df["LS_BLS_Unemployment_Rate_Change_12M_State_Fallback"])
)

# No LS_BLS_Employment_Final / LS_BLS_Labor_Force_Final is created:
# State totals are not comparable replacements for MSA/MSAD totals.

print("Strict PIT final coverage:")
display(
    pd.DataFrame({
        "raw_missing": [
            df["LS_BLS_Unemployment_Rate"].isna().sum(),
            df["LS_BLS_Employment_Growth_12M"].isna().sum(),
            df["LS_BLS_Unemployment_Rate_Change_12M"].isna().sum(),
        ],
        "final_missing": [
            df["LS_BLS_Unemployment_Rate_Final"].isna().sum(),
            df["LS_BLS_Employment_Growth_12M_Final"].isna().sum(),
            df["LS_BLS_Unemployment_Rate_Change_12M_Final"].isna().sum(),
        ],
    }, index=[
        "Unemployment Rate",
        "Employment Growth 12M",
        "Unemployment Rate Change 12M"
    ])
)


Strict PIT final coverage:


,raw_missing,final_missing
Unemployment Rate,128686,333
Employment Growth 12M,128686,333
Unemployment Rate Change 12M,128686,333


## 6. Coverage audit: Regular vs Strict


In [9]:
coverage_pairs = [
    ("Regular", "Unemployment Rate",
     "BLS_Unemployment_Rate", "BLS_Unemployment_Rate_Final"),
    ("Regular", "Employment Growth 12M",
     "BLS_Employment_Growth_12M", "BLS_Employment_Growth_12M_Final"),
    ("Regular", "Unemployment Rate Change 12M",
     "BLS_Unemployment_Rate_Change_12M", "BLS_Unemployment_Rate_Change_12M_Final"),
    ("Strict/LS", "Unemployment Rate",
     "LS_BLS_Unemployment_Rate", "LS_BLS_Unemployment_Rate_Final"),
    ("Strict/LS", "Employment Growth 12M",
     "LS_BLS_Employment_Growth_12M", "LS_BLS_Employment_Growth_12M_Final"),
    ("Strict/LS", "Unemployment Rate Change 12M",
     "LS_BLS_Unemployment_Rate_Change_12M", "LS_BLS_Unemployment_Rate_Change_12M_Final"),
]

rows = []
for timeline, signal, raw, final in coverage_pairs:
    rows.append({
        "timeline": timeline,
        "signal": signal,
        "raw_column": raw,
        "final_column": final,
        "raw_nonmissing": int(df[raw].notna().sum()),
        "raw_missing": int(df[raw].isna().sum()),
        "raw_missing_pct": float(df[raw].isna().mean() * 100),
        "final_nonmissing": int(df[final].notna().sum()),
        "final_missing": int(df[final].isna().sum()),
        "final_missing_pct": float(df[final].isna().mean() * 100),
        "rows_recovered": int((df[raw].isna() & df[final].notna()).sum()),
    })

coverage = pd.DataFrame(rows)
display(coverage)
coverage.to_csv(QA_DIR / "10_bls_regular_strict_coverage_improvement.csv", index=False)

for c in ["BLS_Geography_Source_Final", "LS_BLS_Geography_Source_Final"]:
    summary = (
        df[c].value_counts(dropna=False)
        .rename_axis(c)
        .reset_index(name="rows")
    )
    display(summary)
    summary.to_csv(QA_DIR / f"10_{c}_summary.csv", index=False)


,timeline,signal,raw_column,final_column,raw_nonmissing,raw_missing,raw_missing_pct,final_nonmissing,final_missing,final_missing_pct,rows_recovered
0,Regular,Unemployment Rate,BLS_Unemployment_Rate,BLS_Unemployment_Rate_Final,584343,128657,18.044460,712588,412,0.057784,128245
1,Regular,Employment Growth 12M,BLS_Employment_Growth_12M,BLS_Employment_Growth_12M_Final,584343,128657,18.044460,712588,412,0.057784,128245
2,Regular,Unemployment Rate Change 12M,BLS_Unemployment_Rate_Change_12M,BLS_Unemployment_Rate_Change_12M_Final,584343,128657,18.044460,712588,412,0.057784,128245
3,Strict/LS,Unemployment Rate,LS_BLS_Unemployment_Rate,LS_BLS_Unemployment_Rate_Final,584314,128686,18.048527,712667,333,0.046704,128353
4,Strict/LS,Employment Growth 12M,LS_BLS_Employment_Growth_12M,LS_BLS_Employment_Growth_12M_Final,584314,128686,18.048527,712667,333,0.046704,128353
5,Strict/LS,Unemployment Rate Change 12M,LS_BLS_Unemployment_Rate_Change_12M,LS_BLS_Unemployment_Rate_Change_12M_Final,584314,128686,18.048527,712667,333,0.046704,128353


,BLS_Geography_Source_Final,rows
0,MSA/MSAD_REGULAR,584343
1,STATE_REGULAR,128245
2,UNRESOLVED,412


,LS_BLS_Geography_Source_Final,rows
0,MSA_STRICT_EXISTING,584314
1,STATE_STRICT_FALLBACK,128353
2,MISSING,333


## 7. Missingness by Split and Vintage

This does not impute anything. It checks whether any residual missingness is concentrated in particular modeling periods.


In [10]:
audit_cols = [
    "BLS_Unemployment_Rate_Final",
    "BLS_Employment_Growth_12M_Final",
    "BLS_Unemployment_Rate_Change_12M_Final",
    "LS_BLS_Unemployment_Rate_Final",
    "LS_BLS_Employment_Growth_12M_Final",
    "LS_BLS_Unemployment_Rate_Change_12M_Final",
]

audit_rows = []

for group_cols in [["Split"], ["Dataset_Vintage"], ["Split", "Dataset_Vintage"]]:
    grouped = df.groupby(group_cols, dropna=False)

    for keys, part in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)

        group_info = dict(zip(group_cols, keys))

        for c in audit_cols:
            audit_rows.append({
                "grouping": " × ".join(group_cols),
                **group_info,
                "column": c,
                "rows": len(part),
                "missing": int(part[c].isna().sum()),
                "missing_pct": float(part[c].isna().mean() * 100),
            })

missing_audit = pd.DataFrame(audit_rows)
missing_audit.to_csv(
    QA_DIR / "10_final_bls_missingness_by_split_vintage.csv",
    index=False
)

display(missing_audit.head(30))


,grouping,Split,column,rows,missing,missing_pct,Dataset_Vintage
0,Split,OOT Test,BLS_Unemployment_Rate_Final,99719,26,0.026073,NaN
1,Split,OOT Test,BLS_Employment_Growth_12M_Final,99719,26,0.026073,NaN
2,Split,OOT Test,BLS_Unemployment_Rate_Change_12M_Final,99719,26,0.026073,NaN
3,Split,OOT Test,LS_BLS_Unemployment_Rate_Final,99719,26,0.026073,NaN
4,Split,OOT Test,LS_BLS_Employment_Growth_12M_Final,99719,26,0.026073,NaN
5,Split,OOT Test,LS_BLS_Unemployment_Rate_Change_12M_Final,99719,26,0.026073,NaN
6,Split,Other Period,BLS_Unemployment_Rate_Final,404240,299,0.073966,NaN
7,Split,Other Period,BLS_Employment_Growth_12M_Final,404240,299,0.073966,NaN
8,Split,Other Period,BLS_Unemployment_Rate_Change_12M_Final,404240,299,0.073966,NaN
9,Split,Other Period,LS_BLS_Unemployment_Rate_Final,404240,220,0.054423,NaN


## 8. Hard integrity and lineage assertions

The notebook stops if:

- row count changes,
- `Loan_ID` changes or duplicates,
- target or split changes,
- any original raw BLS column changes,
- a final value fails to preserve a nonmissing local value,
- fallback worsens coverage,
- an invalid State-total replacement feature is created.


In [11]:
# ------------------------------------------------------------
# Dataset integrity
# ------------------------------------------------------------
assert len(df) == n0, "Row count changed."
assert df["Loan_ID"].is_unique, "Loan_ID uniqueness was lost."
assert df["Loan_ID"].equals(loan_id0), "Loan_ID/order changed."
assert df["Target_24M"].equals(target0), "Target_24M changed."
assert df["Split"].equals(split0), "Split changed."
assert set(cols0).issubset(df.columns), "One or more original columns disappeared."

# Original columns must remain value-equivalent.
# Date columns were normalized with pd.to_datetime, so compare them as datetimes.
for c in cols0:
    if c in ["PIT_BLS_Month_Date", "Strict_PIT_BLS_Month_Date", "Strict_PIT_BLS_State_Month_Date"]:
        left = pd.to_datetime(original[c], errors="coerce")
        right = pd.to_datetime(df[c], errors="coerce")
        assert left.equals(right), f"Original date column changed: {c}"
    else:
        assert original[c].equals(df[c]), f"Original column changed: {c}"

# ------------------------------------------------------------
# Final must preserve local value wherever local exists
# ------------------------------------------------------------
final_pairs = [
    ("BLS_Unemployment_Rate", "BLS_Unemployment_Rate_Final"),
    ("BLS_Employment_Growth_12M", "BLS_Employment_Growth_12M_Final"),
    ("BLS_Unemployment_Rate_Change_12M", "BLS_Unemployment_Rate_Change_12M_Final"),
    ("LS_BLS_Unemployment_Rate", "LS_BLS_Unemployment_Rate_Final"),
    ("LS_BLS_Employment_Growth_12M", "LS_BLS_Employment_Growth_12M_Final"),
    ("LS_BLS_Unemployment_Rate_Change_12M", "LS_BLS_Unemployment_Rate_Change_12M_Final"),
]

for raw, final in final_pairs:
    mask = df[raw].notna()
    x = pd.to_numeric(df.loc[mask, raw], errors="coerce")
    y = pd.to_numeric(df.loc[mask, final], errors="coerce")
    assert np.allclose(
        x, y, rtol=0.0, atol=1e-12, equal_nan=True
    ), f"{final} overwrote existing local values from {raw}."

# Coverage cannot worsen.
for _, _, raw, final in coverage_pairs:
    assert df[final].isna().sum() <= df[raw].isna().sum(), (
        f"Fallback worsened coverage: {final}"
    )

# Do not create invalid absolute-total fallbacks.
for forbidden in [
    "BLS_Employment_Final",
    "BLS_Labor_Force_Final",
    "LS_BLS_Employment_Final",
    "LS_BLS_Labor_Force_Final",
]:
    assert forbidden not in df.columns, (
        f"Invalid State-total fallback was created: {forbidden}"
    )

qa = pd.DataFrame([
    ["row_count_preserved", len(df) == n0],
    ["loan_id_unique", df["Loan_ID"].is_unique],
    ["loan_id_preserved", df["Loan_ID"].equals(loan_id0)],
    ["target_preserved", df["Target_24M"].equals(target0)],
    ["split_preserved", df["Split"].equals(split0)],
    ["all_original_columns_preserved", set(cols0).issubset(df.columns)],
    ["strict_state_09G_value_reconstruction_zero_mismatch", int(value_mismatch.sum()) == 0],
    ["strict_state_09G_missingness_reconstruction_zero_mismatch", int(missingness_mismatch.sum()) == 0],
], columns=["check", "passed"])

display(qa)
qa.to_csv(QA_DIR / "10_quality_assurance_checks.csv", index=False)

assert qa["passed"].all(), "At least one hard QA check failed."
print("PASS: all hard QA checks.")


,check,passed
0,row_count_preserved,True
1,loan_id_unique,True
2,loan_id_preserved,True
3,target_preserved,True
4,split_preserved,True
5,all_original_columns_preserved,True
6,strict_state_09G_value_reconstruction_zero_mis...,True
7,strict_state_09G_missingness_reconstruction_ze...,True


PASS: all hard QA checks.


## 9. Remove temporary reconstruction columns and export

Temporary `_REG_...` and `_STRICT_...` lookup columns are dropped. Explicit fallback/final columns and all pre-existing 09G audit fields are retained.


In [12]:
temporary = [
    c for c in df.columns
    if c.startswith("_REG_") or c.startswith("_STRICT_")
]

df_out = df.drop(columns=temporary).copy()

new_cols = [c for c in df_out.columns if c not in cols0]

model_candidates = {
    "BLS_Unemployment_Rate_Final",
    "BLS_Employment_Growth_12M_Final",
    "BLS_Unemployment_Rate_Change_12M_Final",
    "LS_BLS_Employment_Growth_12M_Final",
    "LS_BLS_Unemployment_Rate_Change_12M_Final",
}

manifest = pd.DataFrame({
    "column": new_cols,
    "role": [
        "FINAL_MODEL_CANDIDATE" if c in model_candidates else "LINEAGE_OR_AUDIT"
        for c in new_cols
    ]
})

# Existing LS_BLS_Unemployment_Rate_Final is already a model candidate,
# but it is not "new", so record it explicitly in the manifest.
manifest = pd.concat([
    manifest,
    pd.DataFrame([{
        "column": "LS_BLS_Unemployment_Rate_Final",
        "role": "EXISTING_FINAL_MODEL_CANDIDATE"
    }])
], ignore_index=True)

display(manifest)
manifest.to_csv(QA_DIR / "10_new_column_manifest.csv", index=False)

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
df_out.to_parquet(OUTPUT, index=False)

print("Saved:", OUTPUT)
print(f"Output shape: {df_out.shape[0]:,} rows × {df_out.shape[1]:,} columns")


,column,role
0,BLS_Unemployment_Rate_State_Fallback,LINEAGE_OR_AUDIT
1,BLS_Employment_Growth_12M_State_Fallback,LINEAGE_OR_AUDIT
2,BLS_Unemployment_Rate_Change_12M_State_Fallback,LINEAGE_OR_AUDIT
3,BLS_Unemployment_Rate_Final,FINAL_MODEL_CANDIDATE
4,BLS_Employment_Growth_12M_Final,FINAL_MODEL_CANDIDATE
5,BLS_Unemployment_Rate_Change_12M_Final,FINAL_MODEL_CANDIDATE
6,BLS_State_Fallback_Applied,LINEAGE_OR_AUDIT
7,BLS_Fallback_Level,LINEAGE_OR_AUDIT
8,BLS_Geography_Source_Final,LINEAGE_OR_AUDIT
9,BLS_Remaining_Missing_Reason,LINEAGE_OR_AUDIT


Saved: C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_modeling_dataset_complete_regular_and_strict_BLS_state_fallback.parquet
Output shape: 713,000 rows × 105 columns


## 10. Reload from disk and validate the actual artifact


In [13]:
check = pd.read_parquet(OUTPUT)

post = pd.DataFrame([
    ["saved_file_exists", OUTPUT.exists()],
    ["row_count_preserved", len(check) == n0],
    ["loan_id_unique", check["Loan_ID"].is_unique],
    ["loan_id_preserved", check["Loan_ID"].equals(loan_id0)],
    ["target_preserved", check["Target_24M"].equals(target0)],
    ["split_preserved", check["Split"].equals(split0)],
    ["all_original_columns_present", set(cols0).issubset(check.columns)],
    ["all_new_columns_present", set(new_cols).issubset(check.columns)],
], columns=["check", "passed"])

display(post)
post.to_csv(QA_DIR / "10_post_save_validation.csv", index=False)
assert post["passed"].all(), "Post-save artifact validation failed."

print("\nCOMPLETE")
print("Regular and Strict BLS State fallback lineage has been completed and validated.")
print("Next step: rerun the missing-value audit and feature-set/lineage freeze using the new parquet.")


,check,passed
0,saved_file_exists,True
1,row_count_preserved,True
2,loan_id_unique,True
3,loan_id_preserved,True
4,target_preserved,True
5,split_preserved,True
6,all_original_columns_present,True
7,all_new_columns_present,True



COMPLETE
Regular and Strict BLS State fallback lineage has been completed and validated.
Next step: rerun the missing-value audit and feature-set/lineage freeze using the new parquet.


# Modeling interpretation after this notebook

For a clean Regular-vs-Strict macro experiment, use matched economic concepts.

### Regular PIT final macro candidates

- `Housing_HPI`
- `Housing_HPI_Growth_12M`
- `BLS_Unemployment_Rate_Final`
- `BLS_Employment_Growth_12M_Final`
- `BLS_Unemployment_Rate_Change_12M_Final`

### Strict / leakage-safe PIT final macro candidates

- `LS_Housing_HPI`
- `LS_Housing_HPI_Growth_12M`
- `LS_BLS_Unemployment_Rate_Final`
- `LS_BLS_Employment_Growth_12M_Final`
- `LS_BLS_Unemployment_Rate_Change_12M_Final`

Raw MSA/MSAD variables and State/intermediate variables remain lineage/audit features rather than being mixed indiscriminately with the final resolved representations.
